# Discretização de idade usando somente treinamento

Este notebook avalia apenas a coluna idade e o target da **base de treinamento**. Ele nunca abre uma base de teste. Os cortes por quantis, largura igual e árvore são ajustados novamente dentro de cada dobra de validação interna do treino. Depois da escolha, os cutpoints são ajustados no treino completo e congelados.

**Uso com dados reais:** no próximo bloco, defina o caminho de um arquivo que contenha somente treino e altere os nomes das colunas e limites na configuração. O exemplo sintético padrão serve apenas para verificar o fluxo; seus cutpoints não devem ser reutilizados.

A escolha aplica primeiro os mínimos de total, performance, bons e maus, cobertura e estabilidade. Entre os esquemas viáveis, compara o ganho de log-loss da idade isolada em validação interna e escolhe o menor número de faixas dentro de um erro padrão do melhor ganho. O target de outras features candidatas não é consultado.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from analise_idade_treino import (
    ConfigIdade, preparar_treino, resumo_qualidade, visualizar_exploracao,
    comparar_esquemas, escolher_esquema, visualizar_comparacao,
    visualizar_escolhido, especificacao_final, aplicar_cortes,
    salvar_especificacao,
)

# Defina um caminho real para analisar seus dados. None executa somente a demonstração.
CAMINHO_TREINO = None

if CAMINHO_TREINO is None:
    rng = np.random.default_rng(7)
    n = 18_000
    idade = rng.integers(18, 76, n).astype(float)
    safra = np.repeat(["2024S1", "2024S2", "2025S1"], n // 3)
    prob_mau = 1 / (1 + np.exp(-(-2.4 + 0.035 * (idade - 35) + 0.001 * (idade - 45) ** 2)))
    mau = rng.binomial(1, prob_mau).astype(float)
    idade[rng.random(n) < 0.01] = np.nan
    idade[rng.choice(n, 20, replace=False)] = -1
    mau[rng.random(n) < 0.03] = np.nan
    df_treino = pd.DataFrame({
        "idade": idade, "mau": mau, "safra": safra, "particao": "treino"
    })
    print("DEMONSTRAÇÃO SINTÉTICA — nenhum resultado representa seus dados.")
else:
    df_treino = pd.read_csv(Path(CAMINHO_TREINO))
    print(f"Arquivo de treinamento carregado: {len(df_treino):,} linhas.")


## 1. Configuração e barreira contra vazamento

Preencha os limites conforme a política de cohorts. Se o arquivo tiver uma coluna que marca treino/teste, configure particao_col e valor_treino: a função rejeita qualquer linha que não seja de treinamento. Se o arquivo já contiver somente treino e não tiver esse indicador, use particao_col=None e mantenha o isolamento na preparação do arquivo.

Se houver safras, informe a ordem **cronológica**. As dobras internas usarão períodos anteriores para ajustar cortes e o período seguinte para avaliar. Sem safra, a validação interna usa dobras aleatórias dentro do treino.


In [ ]:
cfg = ConfigIdade(
    idade_col="idade",
    target_col="mau",
    safra_col="safra",  # troque por None se não houver tempo
    ordem_safras=("2024S1", "2024S2", "2025S1"),  # troque pelos períodos do seu treino
    particao_col="particao",  # troque por None se o arquivo já vier filtrado
    valor_treino="treino",
    idade_min=0,
    idade_max=120,
    k_min=2,
    k_max=6,
    min_total=1000,
    min_performance=850,
    min_bons=650,
    min_maus=100,
    cobertura_minima=0.80,
    presenca_temporal_minima=1.0,
    desvio_relativo_maximo=0.75,
    cv_dobras=5,
    semente=20261105,
)

base = preparar_treino(df_treino, cfg)
print(f"Treino validado: {len(base):,} registros.")


## 2. Qualidade e distribuição da idade

A tabela mostra quantos registros têm idade válida, ausente ou inválida e os respectivos bons/maus. O histograma mostra a concentração das idades. O segundo gráfico apresenta a bad rate por idade inteira e seu intervalo de Wilson; pontos com poucos eventos terão intervalos largos. Com safras, o terceiro gráfico mostra a evolução de idades válidas, ausentes e inválidas. Esta exploração usa somente treino.


In [ ]:
display(resumo_qualidade(base, cfg))
fig_exploracao, tabela_idade = visualizar_exploracao(base, cfg)
display(tabela_idade.head(30))
plt.show()


## 3. Comparação dos cortes

São gerados cortes por quantis, intervalos de largura igual e árvore de uma variável. Cada candidato precisa passar nos mínimos de contagem e estabilidade. A coluna ganho_logloss_cv mede quanto a idade discretizada melhora a previsão em relação à bad rate constante de cada dobra **dentro do treino**. Valor positivo é melhor; este sinal é complementar aos filtros estruturais.


In [ ]:
comparacao, detalhes = comparar_esquemas(base, cfg)
colunas = [
    "candidato", "n_faixas", "cortes", "viavel", "cobertura",
    "menor_total", "menor_performance", "menor_bons", "menor_maus",
    "presenca_minima", "desvio_relativo_maximo",
    "ganho_logloss_cv", "erro_padrao_cv", "dobras_cv"
]
display(comparacao[colunas])
escolhido = escolher_esquema(comparacao)
display(escolhido.to_frame("escolhido"))
fig_comparacao = visualizar_comparacao(comparacao, escolhido)
plt.show()


## 4. Inspeção da solução escolhida

Confira a posição dos cutpoints na distribuição, as contagens e bad rates por faixa e a participação de cada faixa nas safras. A tabela por faixa deve ser revisada antes de incorporar faixa_idade ao cubo; a escolha univariada não garante viabilidade depois de cruzá-la com as demais dimensões.


In [ ]:
fig_final = visualizar_escolhido(base, cfg, escolhido, detalhes)
display(detalhes[escolhido.candidato]["faixas"])
if not detalhes[escolhido.candidato]["temporal"].empty:
    display(detalhes[escolhido.candidato]["temporal"])
plt.show()

treino_com_faixa = df_treino.copy()
treino_com_faixa["faixa_idade"] = aplicar_cortes(
    base, tuple(escolhido.cortes)
).to_numpy()
display(treino_com_faixa["faixa_idade"].value_counts(dropna=False).to_frame("qtd_total"))


## 5. Congelamento dos limites

A especificação registra cutpoints e a convenção dos intervalos: um valor exatamente igual ao cutpoint pertence à faixa superior. Ausentes e inválidos recebem categorias próprias. Salve o JSON com uma versão nova depois da revisão. Para usar esses limites em dados futuros, aplique-os como estão; **não ajuste novamente os cortes no teste**.

O catálogo de cohorts poderá então receber faixa_idade como mais uma dimensão. O número de combinações será 2^N−1 para o novo total de dimensões.


In [ ]:
especificacao = especificacao_final(escolhido, cfg)
print(json.dumps(especificacao, ensure_ascii=False, indent=2, default=str))

# Descomente apenas após revisar o resultado e escolher um nome de versão.
# salvar_especificacao(especificacao, "saidas/faixas_idade_v1.json")
